# CPO_19: Extract → Inspect → OCR → Inspect Raw

Notebook này **không tự triển khai OCR mới**. Nó điều khiển pipeline hiện có trong repository bằng đúng CLI:

```bash
python -m src.pipeline --config configs/CPO_19.json --steps extract
python -m src.pipeline --config configs/CPO_19.json --steps ocr
```

Luồng được thiết kế theo README dự án:
1. Extract PDF thành `02_extraction/pages/NNNN.png`.
2. Kiểm tra số lượng, tên, kích thước và xem ảnh trước khi OCR.
3. Chạy Qwen Single-Stream OCR thông qua code hiện có.
4. Kiểm tra raw JSON, `ocr_raw/raw.xml`, `ocr_raw_text/NNNN.txt`, page coverage và line IDs.

**An toàn dữ liệu:** không dùng `--force`; notebook không sửa hoặc xóa OCR artifacts hiện có. Nếu output đã tồn tại, pipeline có thể bỏ qua hoặc báo lỗi theo logic hiện hành. Hãy kiểm tra trước khi quyết định chạy lại.

## 0. Cấu hình repository

In [ ]:
from pathlib import Path
import os, sys, json, subprocess, re, hashlib
from collections import Counter
from IPython.display import display, Image as IPyImage
import pandas as pd

# SỬA đường dẫn này thành thư mục gốc repository.
# Ví dụ Windows: r"E:\Projects\han-viet-corpus"
# Ví dụ Linux/Colab: "/content/han-viet-corpus"
PROJECT_ROOT = Path(r"E:\Projects\han-viet-corpus")

CONFIG_REL = Path("configs/CPO_19.json")
WORK_ID = "CPO_19"
DATA_ROOT = Path("data")

CONFIG_PATH = PROJECT_ROOT / CONFIG_REL
WORK_DIR = PROJECT_ROOT / DATA_ROOT / WORK_ID

assert PROJECT_ROOT.exists(), f"Không tìm thấy repository: {PROJECT_ROOT}"
assert (PROJECT_ROOT / "src" / "pipeline.py").exists(), "Không tìm thấy src/pipeline.py"
assert CONFIG_PATH.exists(), f"Không tìm thấy config: {CONFIG_PATH}"

print("Repository:", PROJECT_ROOT)
print("Config:", CONFIG_PATH)
print("Work directory:", WORK_DIR)
print("Python:", sys.executable)

## 1. Đọc và kiểm tra config trước khi chạy

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
common = config.get("common", {})
ocr_cfg = config.get("ocr", {})

print(json.dumps({
    "work_id": common.get("work_id"),
    "data_root": common.get("data_root", "data"),
    "source": common.get("source"),
    "pages": common.get("pages"),
    "dpi": common.get("dpi"),
    "ocr": ocr_cfg
}, ensure_ascii=False, indent=2))

assert common.get("work_id") == WORK_ID, (
    f"Config work_id phải là {WORK_ID}, hiện tại là {common.get('work_id')}"
)
assert common.get("source"), "Config thiếu common.source"
assert common.get("pages"), "Config thiếu common.pages"
assert ("engine" in ocr_cfg or "engines" in ocr_cfg
        or any(k in ocr_cfg for k in ("chi_tra", "vie"))), (
    "Không nhận diện được cấu trúc ocr trong config. Kiểm tra configs/CPO_19.json."
)

## 2. Helper chạy pipeline và hiển thị output

In [ ]:
def run_pipeline(steps, dry_run=False):
    cmd = [
        sys.executable, "-m", "src.pipeline",
        "--config", str(CONFIG_REL.as_posix()),
        "--steps", steps
    ]
    if dry_run:
        cmd.append("--dry-run")

    print("$", subprocess.list2cmdline(cmd))
    result = subprocess.run(
        cmd,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf-8",
        errors="replace"
    )
    print(result.stdout)
    print("Exit code:", result.returncode)
    if result.returncode != 0:
        raise RuntimeError(f"Pipeline failed at steps={steps!r}, exit={result.returncode}")
    return result.stdout

def list_files(folder, patterns=("*.png",)):
    folder = Path(folder)
    found = []
    if folder.exists():
        for pattern in patterns:
            found.extend(folder.glob(pattern))
    return sorted(set(found))

## 3. Dry-run Extract: xác nhận lệnh và đường dẫn

In [ ]:
extract_plan = run_pipeline("extract", dry_run=True)

## 4. Chạy Extract

In [ ]:
# Không thêm --force. Giữ nguyên artifacts đang có.
extract_log = run_pipeline("extract", dry_run=False)

## 5. Kiểm tra ảnh đã extract

In [ ]:
PAGES_DIR = WORK_DIR / "02_extraction" / "pages"
page_images = list_files(PAGES_DIR, ("*.png", "*.jpg", "*.jpeg", "*.tif", "*.tiff"))

print("Pages directory:", PAGES_DIR)
print("Extracted image count:", len(page_images))
print("First files:", [p.name for p in page_images[:10]])
print("Last files:", [p.name for p in page_images[-5:]])

# Kiểm tra naming convention 4 chữ số.
bad_names = [p.name for p in page_images if not re.match(r"^\d{4}\.[^.]+$", p.name)]
print("Non 4-digit filenames:", bad_names[:20])

image_rows = []
try:
    from PIL import Image
    for path in page_images:
        try:
            with Image.open(path) as im:
                image_rows.append({
                    "filename": path.name,
                    "width": im.width,
                    "height": im.height,
                    "mode": im.mode,
                    "bytes": path.stat().st_size
                })
        except Exception as e:
            image_rows.append({"filename": path.name, "error": repr(e)})

    image_df = pd.DataFrame(image_rows)
    display(image_df.head(30))
except ImportError:
    print("Pillow chưa được cài. Cài bằng: pip install pillow")

## 6. Xem ảnh để kiểm tra chất lượng trước OCR

In [ ]:
# Chọn các trang đầu, giữa, cuối trong phạm vi đã extract.
if page_images:
    sample_indices = sorted(set([0, len(page_images)//2, len(page_images)-1]))
    for idx in sample_indices:
        path = page_images[idx]
        print(path.name)
        display(IPyImage(filename=str(path)))
else:
    print("Không tìm thấy ảnh. Kiểm tra common.pages, data_root và output extract.")

### Checklist trước khi OCR

- [ ] Số ảnh đúng với phạm vi `common.pages`.
- [ ] Tên file có dạng `NNNN.png`.
- [ ] Không có ảnh lỗi, trắng bất thường hoặc thiếu trang.
- [ ] Hướng đọc, chiều trang và crop hợp lý.
- [ ] Chữ Hán và chữ Quốc ngữ đủ rõ để nhận dạng.

Chỉ chạy cell OCR sau khi đã kiểm tra ảnh.

## 7. Dry-run OCR: xác nhận Qwen Single-Stream

In [ ]:
ocr_plan = run_pipeline("ocr", dry_run=True)

## 8. Chạy OCR bằng code hiện có

In [ ]:
# Chạy OCR theo configs/CPO_19.json. Không dùng --force.
ocr_log = run_pipeline("ocr", dry_run=False)

## 9. Kiểm tra cấu trúc OCR artifacts

In [ ]:
OCR_DIR = WORK_DIR / "03_ocr"
RAW_JSON_DIR = OCR_DIR / "raw"
UNIFIED_XML = OCR_DIR / "ocr_raw" / "raw.xml"
PAGE_TEXT_DIR = OCR_DIR / "ocr_raw_text"

raw_json_files = sorted(RAW_JSON_DIR.glob("*.json")) if RAW_JSON_DIR.exists() else []
page_text_files = sorted(PAGE_TEXT_DIR.glob("*.txt")) if PAGE_TEXT_DIR.exists() else []

print("Raw JSON dir:", RAW_JSON_DIR, "| files:", len(raw_json_files))
print("Unified XML:", UNIFIED_XML, "| exists:", UNIFIED_XML.exists())
print("Per-page text dir:", PAGE_TEXT_DIR, "| files:", len(page_text_files))
print("Raw JSON examples:", [p.name for p in raw_json_files[:10]])
print("TXT examples:", [p.name for p in page_text_files[:10]])

## 10. Phân tích raw JSON theo trang

In [ ]:
def load_json_safely(path):
    try:
        return json.loads(path.read_text(encoding="utf-8-sig"))
    except Exception as e:
        return {"__load_error__": repr(e)}

raw_rows = []
raw_errors = []
for path in raw_json_files:
    obj = load_json_safely(path)
    if "__load_error__" in obj:
        raw_errors.append({"file": path.name, "error": obj["__load_error__"]})
        continue

    # Hỗ trợ báo cáo nhiều schema phổ biến, không sửa file nguồn.
    lines = obj.get("lines")
    if lines is None:
        lines = obj.get("items")
    if lines is None and isinstance(obj.get("result"), dict):
        lines = obj["result"].get("lines", obj["result"].get("items", []))
    if not isinstance(lines, list):
        lines = []

    texts = []
    for item in lines:
        if isinstance(item, dict):
            texts.append(str(item.get("text", item.get("text_raw", ""))))
        elif isinstance(item, str):
            texts.append(item)

    raw_rows.append({
        "file": path.name,
        "line_count": len(lines),
        "nonempty_lines": sum(bool(t.strip()) for t in texts),
        "character_count": sum(len(t) for t in texts),
        "has_text": any(t.strip() for t in texts),
        "top_level_keys": ", ".join(obj.keys())
    })

raw_df = pd.DataFrame(raw_rows)
display(raw_df.head(40))
print("JSON parse errors:", len(raw_errors))
if raw_errors:
    display(pd.DataFrame(raw_errors))

## 11. Kiểm tra unified raw.xml và line IDs

In [ ]:
import xml.etree.ElementTree as ET

xml_report = {"exists": UNIFIED_XML.exists()}
xml_lines = []
xml_error = None

if UNIFIED_XML.exists():
    try:
        tree = ET.parse(UNIFIED_XML)
        root = tree.getroot()
        print("Root tag:", root.tag)
        print("Root attributes:", root.attrib)

        # Dò các phần tử line theo tag name không phân biệt hoa thường.
        for elem in root.iter():
            if elem.tag.split("}")[-1].lower() == "line":
                xml_lines.append(elem)

        print("XML line elements:", len(xml_lines))
        for elem in xml_lines[:5]:
            print(elem.tag, elem.attrib, (elem.text or "")[:200])
    except Exception as e:
        xml_error = repr(e)
        print("XML parse error:", xml_error)
else:
    print("raw.xml chưa tồn tại. Kiểm tra log OCR và cấu hình module OCR.")

# Kiểm tra line_id theo quy ước PNNNN.LNNN nếu thuộc tính hiện diện.
line_ids = []
for elem in xml_lines:
    line_id = elem.attrib.get("id") or elem.attrib.get("line_id")
    if line_id:
        line_ids.append(line_id)

bad_line_ids = [x for x in line_ids if not re.match(r"^P\d{4}\.L\d{3,}$", x)]
print("Line IDs found:", len(line_ids))
print("Nonconforming line IDs:", bad_line_ids[:20])

## 12. Kiểm tra page coverage và đối chiếu artifacts

In [ ]:
def page_number_from_filename(path):
    m = re.match(r"^(\d{4})", Path(path).name)
    return int(m.group(1)) if m else None

extracted_pages = {page_number_from_filename(p) for p in page_images}
raw_pages = {page_number_from_filename(p) for p in raw_json_files}
txt_pages = {page_number_from_filename(p) for p in page_text_files}

extracted_pages.discard(None)
raw_pages.discard(None)
txt_pages.discard(None)

coverage = pd.DataFrame([
    {"artifact":"extracted images", "count":len(extracted_pages)},
    {"artifact":"raw JSON pages", "count":len(raw_pages)},
    {"artifact":"per-page TXT", "count":len(txt_pages)}
])
display(coverage)

print("Extracted but no raw JSON:", sorted(extracted_pages - raw_pages))
print("Raw JSON but no extracted image:", sorted(raw_pages - extracted_pages))
print("Raw JSON but no TXT:", sorted(raw_pages - txt_pages))
print("TXT but no raw JSON:", sorted(txt_pages - raw_pages))

## 13. Kiểm tra nhanh nội dung OCR theo trang

In [ ]:
def preview_page_json(page_no, max_lines=40):
    path = RAW_JSON_DIR / f"{page_no:04d}.json"
    if not path.exists():
        print("Không có:", path)
        return
    obj = load_json_safely(path)
    print("File:", path.name)
    print("Top-level keys:", list(obj.keys()))

    lines = obj.get("lines", obj.get("items", []))
    if not isinstance(lines, list):
        print(json.dumps(obj, ensure_ascii=False, indent=2)[:5000])
        return

    for i, item in enumerate(lines[:max_lines], 1):
        if isinstance(item, dict):
            print(i, item.get("line_id", item.get("id", "")),
                  item.get("text", item.get("text_raw", "")))
        else:
            print(i, item)

# Thay 15 bằng trang muốn kiểm tra.
preview_page_json(15)

## 14. Kiểm tra văn bản tổng hợp theo trang

In [ ]:
def preview_page_txt(page_no):
    path = PAGE_TEXT_DIR / f"{page_no:04d}.txt"
    if not path.exists():
        print("Không có:", path)
        return
    print(path.read_text(encoding="utf-8"))

preview_page_txt(15)

## 15. Báo cáo kiểm tra cuối

In [ ]:
summary = {
    "work_id": WORK_ID,
    "extract_image_count": len(page_images),
    "raw_json_page_count": len(raw_json_files),
    "page_txt_count": len(page_text_files),
    "raw_xml_exists": UNIFIED_XML.exists(),
    "raw_xml_line_count": len(xml_lines),
    "raw_json_parse_errors": len(raw_errors),
    "xml_parse_error": xml_error,
    "extracted_without_raw": sorted(extracted_pages - raw_pages),
    "raw_without_extract": sorted(raw_pages - extracted_pages),
    "raw_without_txt": sorted(raw_pages - txt_pages),
    "bad_line_ids_sample": bad_line_ids[:20]
}
print(json.dumps(summary, ensure_ascii=False, indent=2))

## 16. Quy tắc an toàn và bước tiếp theo

- Notebook chỉ gọi `extract` và `ocr`; chưa chạy segmentation, cleaning, sentence segmentation, validation hay alignment.
- Không thêm `--force`. Nếu cần OCR lại một trang, hãy xác định rõ cơ chế force/overwrite trong `src/ocr/ocr.py` và backup artifacts trước.
- Không sửa `03_ocr/raw/NNNN.json` hoặc `03_ocr/ocr_raw/raw.xml` bằng notebook.
- Nếu raw JSON có line nhưng XML thiếu line, cần audit bước normalization thay vì chạy OCR lại.
- Nếu `raw.xml` hợp lệ, mới chuyển sang notebook/step kiểm tra document segmentation và layer segmentation.